# 03_MODELADO

En este script utilizaremos distintos modelos de clasificación, y buscaremos el mejor de ellos. En nuestro caso, deberemos tener en cuenta que un falso negativo es mucho mas grave que un falso positivo, luego buscaremos Maximizar el Recall sobre la Precision

In [ ]:
#Lectura de los dataset y volvemos a separar
import pandas as pd
train=pd.read_csv(r"\Documentos\HEART\data\processed\train.csv") #cambiar a valor propio del ordenador.
test=pd.read_csv(r"\HEART\data\processed\test.csv")
X_train=train.drop(columns='target')
y_train=train['target']
X_test=test.drop(columns='target')
y_test=test['target']

In [9]:
#Definición del preprocesador, esto nos ayuda a que en caso de que haya que hacer cambios de distintos tipos, se lleve a cabo
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

numericas=X_train.drop(columns='Gender').columns.tolist()
preprocessor= ColumnTransformer([("num",StandardScaler(), numericas)],remainder="passthrough")

In [10]:
#A continuación, realizaremos distintas pipelines con distintos modelos
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
pipeline_lr=Pipeline([("prep",preprocessor),("model",LogisticRegression(max_iter=1000))])
pipeline_rf=Pipeline([("prep",preprocessor),("model",RandomForestClassifier(random_state=42))])
pipeline_xg=Pipeline([("prep",preprocessor),("model",XGBClassifier(random_state=42,eval_metric='logloss'))])

In [11]:
#A continuación, utilizaremos validación cruzada para ver que modelo es el mejor
from sklearn.model_selection import cross_validate, StratifiedKFold

val=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring=['accuracy','recall','roc_auc']

for name, pipe in [("RegreLog",pipeline_lr),("Random Forest",pipeline_rf),("XGBoost",pipeline_xg)]:
    scores= cross_validate(pipe,X_train,y_train,cv=val,scoring=scoring)
    print(f"{name}")
    for metric in scoring:
        values=scores[f"test_{metric}"]
        print(f"{metric}: {values.mean()}:.2f +- {values.std()}:.2f")

RegreLog
accuracy: 0.7850778605280976:.2f +- 0.051180259366476834:.2f
recall: 0.8342874180083483:.2f +- 0.033376135124120236:.2f
roc_auc: 0.869840797354505:.2f +- 0.03849691788285139:.2f
Random Forest
accuracy: 0.9857413676371023:.2f +- 0.003011875649889252:.2f
recall: 0.9814311270125223:.2f +- 0.006175073059142538:.2f
roc_auc: 0.996899977142619:.2f +- 0.0033682400511260155:.2f
XGBoost
accuracy: 0.989550891446626:.2f +- 0.006983863796137186:.2f
recall: 0.9891830649970185:.2f +- 0.009254775032688619:.2f
roc_auc: 0.9969834936671769:.2f +- 0.003964873989484099:.2f


Finalmente nos quedaremos con el modelo XGBoost por tener mayores métricas en los tres. En el archivo 04_optimización, buscaremos modificar los paramteros del XGboost de manera independiente para maximizar las métricas.